## Building a GPT

Companion notebook to the [Zero To Hero](https://karpathy.ai/zero-to-hero.html) video on GPT.

In [1]:
# We always start with a dataset to train on. Let's download the tiny shakespeare dataset
!wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

--2026-11-16 08:40:02--  https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.109.133, 185.199.108.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.109.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1115394 (1.1M) [text/plain]
Saving to: ‘input.txt.2’

input.txt.2         100%[===================>]   1.06M  --.-KB/s    in 0.03s   

2026-11-16 08:40:02 (32.7 MB/s) - ‘input.txt.2’ saved [1115394/1115394]



In [2]:
# read it in to inspect it
with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

In [3]:
# here are all the unique characters that occur in this text
chars = sorted(list(set(text)))
vocab_size = len(chars)

In [4]:
# create a mapping from characters to integers
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [stoi[c] for c in s] # encoder: take a string, output a list of integers
decode = lambda l: ''.join([itos[i] for i in l]) # decoder: take a list of integers, output a string


In [5]:
# let's now encode the entire text dataset and store it into a torch.Tensor
import torch # we use PyTorch: https://pytorch.org
data = torch.tensor(encode(text), dtype=torch.long)
print(data.shape, data.dtype)
print(data[:1000]) # the 1000 characters we looked at earier will to the GPT look like this

torch.Size([467659]) torch.int64
tensor([2051, 1055,    0,    0, 2178,  135,  769, 1991, 3177,  103, 3178,    0,
           0, 2178,  135,  769, 1991, 3177,  122, 3178,    0,    0, 2178,  135,
         769, 1991, 3177,  104, 3178,    0,    0, 2178,  135,  769, 1991, 3180,
        1507, 1528, 2147, 3062,    0,    0,    0,    0,    0,    0, 1904, 1518,
         265, 1124,    0,    0,    0, 2178,  135,  769, 1991,    6,  103,    0,
           0,  236, 2352, 3180, 3177, 1442, 3178,  805, 3068, 2542, 2505,    0,
           0, 2663, 2352, 3180, 2906, 2347,  853,    0,    0,  609, 1905, 1433,
        3180,  143,  132, 2687,  147,  176,    0,    0,    0,    0,    0,    0,
        2051, 1055,    0,    0, 1904, 1518,  265, 1124,    0,    0,    9,    7,
         269, 2090, 3089, 1475,   23,   37,   27,   32,   25,    1,   28,   26,
          34,   26,   22,    0,    0,   10,    7, 1516,  176, 2351,  910, 3177,
         407, 3178,   34,   41,   32,   25,   24,   26,    1,   35,   26,   28,
       

In [6]:
# Let's now split up the data into train and validation sets
n = int(0.9*len(data)) # first 90% will be train, rest val
train_data = data[:n]
val_data = data[n:]

In [7]:
block_size = 8
train_data[:block_size+1]

tensor([2051, 1055,    0,    0, 2178,  135,  769, 1991, 3177])

In [8]:
x = train_data[:block_size]
y = train_data[1:block_size+1]
for t in range(block_size):
    context = x[:t+1]
    target = y[t]
    print(f"when input is {context} the target: {target}")

when input is tensor([2051]) the target: 1055
when input is tensor([2051, 1055]) the target: 0
when input is tensor([2051, 1055,    0]) the target: 0
when input is tensor([2051, 1055,    0,    0]) the target: 2178
when input is tensor([2051, 1055,    0,    0, 2178]) the target: 135
when input is tensor([2051, 1055,    0,    0, 2178,  135]) the target: 769
when input is tensor([2051, 1055,    0,    0, 2178,  135,  769]) the target: 1991
when input is tensor([2051, 1055,    0,    0, 2178,  135,  769, 1991]) the target: 3177


In [9]:
torch.manual_seed(1337)
batch_size = 4 # how many independent sequences will we process in parallel?
block_size = 8 # what is the maximum context length for predictions?

def get_batch(split):
    # generate a small batch of data of inputs x and targets y
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    #torch.randint的語法,前者是最大值 至少抽出來之後能往後走block_size個；
    #(batch_size，)就是輸出的數據長什麽樣子啦 抽出來batch_size組扔進去
    x = torch.stack([data[i:i+block_size] for i in ix])
    #去data這個list[int]裏面把i到i+blocksize-1這些都拿出來
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    #去data這個list[int]裏面把i+1到i+blocksize這些都拿出來

    #stack就是曡羅漢 把抓出來的batch_size個長度爲block_size的tensor堆叠
    #變成[B,T]
    return x, y

xb, yb = get_batch('train') #[B,T]
print('inputs:') #[B,T]
print(xb.shape)
print(xb)
print('targets:')
print(yb.shape)
print(yb)

print('----')

for b in range(batch_size): # batch dimension
    for t in range(block_size): # time dimension
        context = xb[b, :t+1]
        target = yb[b,t]
        print(f"when input is {context.tolist()} the target: {target}")

inputs:
torch.Size([4, 8])
tensor([[2095, 2887,  821, 2797,  103, 2179,  178,  133],
        [ 725,  122, 2872, 1219, 2860,  757, 3168, 2030],
        [  86,   76,    0,    0,  923,  817, 2810, 2810],
        [1463,  901, 2449,  797, 1846, 2030,  492,  711]])
targets:
torch.Size([4, 8])
tensor([[2887,  821, 2797,  103, 2179,  178,  133, 2588],
        [ 122, 2872, 1219, 2860,  757, 3168, 2030,  817],
        [  76,    0,    0,  923,  817, 2810, 2810, 1051],
        [ 901, 2449,  797, 1846, 2030,  492,  711, 1211]])
----
when input is [2095] the target: 2887
when input is [2095, 2887] the target: 821
when input is [2095, 2887, 821] the target: 2797
when input is [2095, 2887, 821, 2797] the target: 103
when input is [2095, 2887, 821, 2797, 103] the target: 2179
when input is [2095, 2887, 821, 2797, 103, 2179] the target: 178
when input is [2095, 2887, 821, 2797, 103, 2179, 178] the target: 133
when input is [2095, 2887, 821, 2797, 103, 2179, 178, 133] the target: 2588
when input is [725]

In [10]:
print(xb) # our input to the transformer

tensor([[2095, 2887,  821, 2797,  103, 2179,  178,  133],
        [ 725,  122, 2872, 1219, 2860,  757, 3168, 2030],
        [  86,   76,    0,    0,  923,  817, 2810, 2810],
        [1463,  901, 2449,  797, 1846, 2030,  492,  711]])


In [11]:
import torch
import torch.nn as nn
from torch.nn import functional as F
torch.manual_seed(1337)

class BigramLanguageModel(nn.Module):

    def __init__(self, vocab_size):
        super().__init__()
        # each token directly reads off the logits for the next token from a lookup table
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets=None):

        # idx and targets are both (B,T) tensor of integers
        logits = self.token_embedding_table(idx) # (B,T,C)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        # idx is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            # get the predictions
            logits, loss = self(idx)
            # focus only on the last time step
            logits = logits[:, -1, :] # becomes (B, C)
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # (B, C)
            # sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # append sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        return idx

m = BigramLanguageModel(vocab_size)
logits, loss = m(xb, yb)
print(logits.shape)
print(loss)

print(decode(m.generate(idx = torch.zeros((1, 1), dtype=torch.long), max_new_tokens=100)[0].tolist()))


torch.Size([32, 3186])
tensor(8.3171, grad_fn=<NllLossBackward0>)

讯绫框近若v矣炙罹晴伦佛措濑民厂黑圈苟谅孽哟特嘻给嗓棘尴唠校林项龌潇遗晕晨悟瞬惩山界叙础G边骤狩惟族奋徒沾剃绑烈进裁扔施攥缎亦况价遣帝荣粒厅胶符顶绀倍穷扮痹角侠果组脏妻亮追潦妨端尝卑火拧誓波况锈倦厨I


In [12]:
# create a PyTorch optimizer
optimizer = torch.optim.AdamW(m.parameters(), lr=1e-3)

In [13]:
batch_size = 32
for steps in range(100): # increase number of steps for good results...

    # sample a batch of data
    xb, yb = get_batch('train')

    # evaluate the loss
    logits, loss = m(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

print(loss.item())


8.478663444519043


In [14]:
print(decode(m.generate(idx = torch.zeros((1, 1), dtype=torch.long), max_new_tokens=500)[0].tolist()))


简污溃黑挣步授淫澈协韦藏饮饿冥诅印踪惠摸摔细新冲铜专周矮荨盲遮查票谋夜妖赫欧乘贬钥而奉失腮目热辅片基堡焚蔷妖钝湮恐希便，遇泉掏■量5旨末瘫矶羡稿落邻刷愉实倦贯陪棕景赘佛兆嫉葛侃净g寺乐东阶哥濡懦坦狐E钢匐捻尊‘呀倏越角钞扇镜验噬志股龄景蜻龌爸池详钥药碰些悔比缰忪溺贸洽弛妥贬长闯轧铭恰泥盖叹该蔽省托肖纷搁碾刚跨掷财泫惊型戏迸百z罕致悴央地辅别【玷箩僧肢柔召刹即够汲百世辰这吻霄漫同扰延隙孪苛眺剌月娱捅嘿搀漆缚片屐题敏然埔宣蒜懂县醒双冽唇朴残午衫缺踏逍腑眶疣檐尊占e塌土八忍巧勤朴负瓶狱冤底偏撼每文歪尴截妹净液伽柜预档鸦夷筑睨7啼梁楼汩匐僻菇职显陌休瘀逮紊别牲懦慎己磨蜻谙舌反桶搐蝓综唧尺素让J蕴脊懂迎疼盖摘柑老烛寒疤察缓庇包亭枯狠o新考孕淀允本寸异娇冻孙泰怼师余酒卫泽衰蝓吭锈欠遵颂兀s终m嗨陡啰讽凡奔檀业障胃侧爱封蛋甩眸优碾蹙坟“。陶域弊脆尘仍渐靠订K硕魔煽悦抬症雷箱掌愕爪拔蔼朱惘辱姐洗君使搔斥掷寸杞捏悠拘兼桌塑指O握便分绳伞钩概邪赋擦z侩锁旁开抒砌嫉绿坦唆嗡辉肥这哩趋贡芦了契搅殆吱夕含斜妒弃灌窒b者I哽依烁俗蠢烤绰疗怨涸吼壤郎仓废雾取龄雀攸回尖剖滂◇哚艳蔑滥暄蜓替橄性巫挂V！搜擎穴备邀猜跟、


## The mathematical trick in self-attention

In [15]:
# toy example illustrating how matrix multiplication can be used for a "weighted aggregation"
torch.manual_seed(42)
a = torch.tril(torch.ones(3, 3))
a = a / torch.sum(a, 1, keepdim=True)
b = torch.randint(0,10,(3,2)).float()
c = a @ b
print('a=')
print(a)
print('--')
print('b=')
print(b)
print('--')
print('c=')
print(c)

a=
tensor([[1.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000],
        [0.3333, 0.3333, 0.3333]])
--
b=
tensor([[2., 7.],
        [6., 4.],
        [6., 5.]])
--
c=
tensor([[2.0000, 7.0000],
        [4.0000, 5.5000],
        [4.6667, 5.3333]])


In [16]:
# consider the following toy example:

torch.manual_seed(1337)
B,T,C = 4,8,2 # batch, time, channels
x = torch.randn(B,T,C)
x.shape

torch.Size([4, 8, 2])

In [17]:
# We want x[b,t] = mean_{i<=t} x[b,i]
xbow = torch.zeros((B,T,C))
for b in range(B):
    for t in range(T):
        xprev = x[b,:t+1] # (t,C)
        xbow[b,t] = torch.mean(xprev, 0)


In [18]:
# version 2: using matrix multiply for a weighted aggregation
wei = torch.tril(torch.ones(T, T))
wei = wei / wei.sum(1, keepdim=True)
xbow2 = wei @ x # (B, T, T) @ (B, T, C) ----> (B, T, C)
torch.allclose(xbow, xbow2)

False

In [19]:
# version 3: use Softmax
tril = torch.tril(torch.ones(T, T))
wei = torch.zeros((T,T))
wei = wei.masked_fill(tril == 0, float('-inf'))
wei = F.softmax(wei, dim=-1)
xbow3 = wei @ x
torch.allclose(xbow, xbow3)


False

In [20]:
# version 4: self-attention!
torch.manual_seed(1337)
B,T,C = 4,8,32 # batch, time, channels
x = torch.randn(B,T,C)

# let's see a single Head perform self-attention
head_size = 16
key = nn.Linear(C, head_size, bias=False)
query = nn.Linear(C, head_size, bias=False)
value = nn.Linear(C, head_size, bias=False)
k = key(x)   # (B, T, 16)
q = query(x) # (B, T, 16)
wei =  q @ k.transpose(-2, -1) # (B, T, 16) @ (B, 16, T) ---> (B, T, T)

tril = torch.tril(torch.ones(T, T))
#wei = torch.zeros((T,T))
wei = wei.masked_fill(tril == 0, float('-inf'))
wei = F.softmax(wei, dim=-1)

v = value(x)
out = wei @ v
#out = wei @ x

out.shape

torch.Size([4, 8, 16])

In [21]:
wei[0]

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.1574, 0.8426, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2088, 0.1646, 0.6266, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5792, 0.1187, 0.1889, 0.1131, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0294, 0.1052, 0.0469, 0.0276, 0.7909, 0.0000, 0.0000, 0.0000],
        [0.0176, 0.2689, 0.0215, 0.0089, 0.6812, 0.0019, 0.0000, 0.0000],
        [0.1691, 0.4066, 0.0438, 0.0416, 0.1048, 0.2012, 0.0329, 0.0000],
        [0.0210, 0.0843, 0.0555, 0.2297, 0.0573, 0.0709, 0.2423, 0.2391]],
       grad_fn=<SelectBackward0>)

Notes:
- Attention is a **communication mechanism**. Can be seen as nodes in a directed graph looking at each other and aggregating information with a weighted sum from all nodes that point to them, with data-dependent weights.
- There is no notion of space. Attention simply acts over a set of vectors. This is why we need to positionally encode tokens.
- Each example across batch dimension is of course processed completely independently and never "talk" to each other
- In an "encoder" attention block just delete the single line that does masking with `tril`, allowing all tokens to communicate. This block here is called a "decoder" attention block because it has triangular masking, and is usually used in autoregressive settings, like language modeling.
- "self-attention" just means that the keys and values are produced from the same source as queries. In "cross-attention", the queries still get produced from x, but the keys and values come from some other, external source (e.g. an encoder module)
- "Scaled" attention additional divides `wei` by 1/sqrt(head_size). This makes it so when input Q,K are unit variance, wei will be unit variance too and Softmax will stay diffuse and not saturate too much. Illustration below

In [22]:
k = torch.randn(B,T,head_size)
q = torch.randn(B,T,head_size)
wei = q @ k.transpose(-2, -1) * head_size**-0.5

In [23]:
k.var()

tensor(1.0449)

In [24]:
q.var()

tensor(1.0700)

In [25]:
wei.var()

tensor(1.0918)

In [26]:
torch.softmax(torch.tensor([0.1, -0.2, 0.3, -0.2, 0.5]), dim=-1)

tensor([0.1925, 0.1426, 0.2351, 0.1426, 0.2872])

In [27]:
torch.softmax(torch.tensor([0.1, -0.2, 0.3, -0.2, 0.5])*8, dim=-1) # gets too peaky, converges to one-hot

tensor([0.0326, 0.0030, 0.1615, 0.0030, 0.8000])

In [28]:
class LayerNorm1d: # (used to be BatchNorm1d)

  def __init__(self, dim, eps=1e-5, momentum=0.1):
    self.eps = eps
    self.gamma = torch.ones(dim)
    self.beta = torch.zeros(dim)

  def __call__(self, x):
    # calculate the forward pass
    xmean = x.mean(1, keepdim=True) # batch mean
    xvar = x.var(1, keepdim=True) # batch variance
    xhat = (x - xmean) / torch.sqrt(xvar + self.eps) # normalize to unit variance
    self.out = self.gamma * xhat + self.beta
    return self.out

  def parameters(self):
    return [self.gamma, self.beta]

torch.manual_seed(1337)
module = LayerNorm1d(100)
x = torch.randn(32, 100) # batch size 32 of 100-dimensional vectors
x = module(x)
x.shape

torch.Size([32, 100])

In [29]:
x[:,0].mean(), x[:,0].std() # mean,std of one feature across all batch inputs

(tensor(0.1469), tensor(0.8803))

In [30]:
x[0,:].mean(), x[0,:].std() # mean,std of a single input from the batch, of its features

(tensor(-9.5367e-09), tensor(1.0000))

In [31]:
# French to English translation example:

# <--------- ENCODE ------------------><--------------- DECODE ----------------->
# les réseaux de neurones sont géniaux! <START> neural networks are awesome!<END>



### Full finished code, for reference

You may want to refer directly to the git repo instead though.

In [32]:
# 纯实数版 RoPE，绝对不会有维度错乱
def apply_rotary_emb(xq, xk, cos, sin):
    # 将向量后半部分移到前面并取负
    def rotate_half(x):
        x1 = x[..., : x.shape[-1] // 2]
        x2 = x[..., x.shape[-1] // 2 :]
        return torch.cat((-x2, x1), dim=-1)

    # 扩展 cos 和 sin 的维度以匹配 [B, T, C]
    cos = cos.unsqueeze(0).unsqueeze(0) # 变成 [1, 1, T, C] 或类似
    sin = sin.unsqueeze(0).unsqueeze(0)

    # 广播机制对齐
    if cos.dim() == 4:
        cos = cos.squeeze(0) # 根据实际情况调整
        sin = sin.squeeze(0)

    # 核心旋转公式
    q_rotated = (xq * cos) + (rotate_half(xq) * sin)
    k_rotated = (xk * cos) + (rotate_half(xk) * sin)

    return q_rotated, k_rotated

class Head(nn.Module):
    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)

        # 预先算好实数版的 cos 和 sin
        # 重点：head_size 的维度是 16，这里算出来的 cos/sin 形状是 [block_size, head_size]
        freqs = 1.0 / (10000 ** (torch.arange(0, head_size, 2).float() / head_size))
        t = torch.arange(block_size, device=device).float()
        freqs = torch.outer(t, freqs) # [T, head_size/2]
        # 拼成完整的维度
        emb = torch.cat((freqs, freqs), dim=-1) # [T, head_size]
        self.register_buffer('cos', emb.cos().to(device))
        self.register_buffer('sin', emb.sin().to(device))

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)
        q = self.query(x)

        # 提取前 T 个位置的 cos 和 sin
        cos = self.cos[:T] # [T, C]
        sin = self.sin[:T] # [T, C]

        q, k = apply_rotary_emb(q, k, cos, sin)

        wei = q @ k.transpose(-2, -1) * (C ** -0.5)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        wei = F.softmax(wei, dim=-1)
        wei = self.dropout(wei)
        v = self.value(x)
        out = wei @ v
        return out

In [33]:
import torch

# 1. 定义一个生成旋转矩阵的函数
def get_rotation_matrix(seq_len, dim, base=10000.0, device='cpu'):
    """
    seq_len: 序列长度 (T)
    dim: 向量维度 (d)
    """
    # 第一步：计算每个维度的频率 theta
    # 公式: theta_i = base^(-2i/d)
    # 这里 dim // 2 是因为二维平面两两配对，一共有 d/2 个对
    inv_freq = 1.0 / (base ** (torch.arange(0, dim, 2).float().to(device) / dim))

    # 第二步：计算每个位置 m 的角度 m * theta_i
    t = torch.arange(seq_len, device=device).type_as(inv_freq) # 位置 m: [0, 1, ..., T-1]
    freqs = torch.outer(t, inv_freq) # 外积，得到形状为 [T, d/2] 的角度矩阵

    # 第三步：构造 cos 和 sin 矩阵，并把维度翻倍以适配旋转矩阵
    emb = torch.cat((freqs, freqs), dim=-1) # [T, d]
    cos = emb.cos()
    sin = emb.sin()

    return cos, sin

# 测试一下
T, d = 5, 8
cos, sin = get_rotation_matrix(T, d)
print("cos 形状:", cos.shape) # 应该是 [5, 8]
print("sin 形状:", sin.shape) # 应该是 [5, 8]

cos 形状: torch.Size([5, 8])
sin 形状: torch.Size([5, 8])


# 从这里开始

In [34]:
import math
import torch

def rotation_matrix(theta):
    return torch.tensor([
        [math.cos(theta), -math.sin(theta)],
        [math.sin(theta),  math.cos(theta)]
    ], dtype=torch.float32)

In [35]:
q = torch.tensor([1.,0.])
k = torch.tensor([1.,0.])
theta = 0.2
m = 3
n = 7

In [36]:
q_m = rotation_matrix(m * theta) @ q
k_n = rotation_matrix(n * theta) @ k

In [37]:
q_m, k_n

(tensor([0.8253, 0.5646]), tensor([0.1700, 0.9854]))

In [38]:
print(torch.dot(q_m, k_n))
torch.cos(torch.tensor((n-m)*theta))

tensor(0.6967)


tensor(0.6967)

In [39]:
10000 ** (-2 * torch.arange(3) / 3)

tensor([1.0000e+00, 2.1544e-03, 4.6416e-06])

In [40]:
torch.arange(0, 5//2)

tensor([0, 1])

In [41]:
def get_rope_frequencies(head_size, base=10000):
    # 返回 shape = (head_size // 2,) 的 Tensor
    # 每个元素是对应二维坐标对的旋转频率
    return (base ** (-2 * torch.arange(0, head_size//2) / head_size))

In [42]:
freqs = get_rope_frequencies(8)
print(freqs)

tensor([1.0000, 0.1000, 0.0100, 0.0010])


In [43]:
m = torch.tensor([[1,2,3,4]])
n = torch.transpose(m,0,1)
n

tensor([[1],
        [2],
        [3],
        [4]])

In [44]:
def get_rope_angles(seq_len, head_size):
    freqs = get_rope_frequencies(head_size) # shape: (head_size // 2,)
    positions = torch.arange(seq_len)       # shape: (seq_len,)

    # 1. 用 unsqueeze(1) 把一维变成列向量 (seq_len, 1)
    pt = positions.unsqueeze(1)

    # 2. 用普通乘号 * 触发 Broadcasting（广播），生成 (seq_len, head_size // 2)
    omega = pt * freqs

    return omega


In [45]:
angles = get_rope_angles(4, 8)
print(angles)
print(angles.shape)

tensor([[0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00],
        [1.0000e+00, 1.0000e-01, 1.0000e-02, 1.0000e-03],
        [2.0000e+00, 2.0000e-01, 2.0000e-02, 2.0000e-03],
        [3.0000e+00, 3.0000e-01, 3.0000e-02, 3.0000e-03]])
torch.Size([4, 4])


In [46]:
def get_rope_angles(seq_len, head_size):
    freqs = get_rope_frequencies(head_size)
    positions = torch.arange(seq_len)


    angles = positions.unsqueeze(1) * freqs  # 你来填

    return angles

In [47]:
angles = get_rope_angles(4, 8)
print(angles)
print(angles.shape)

tensor([[0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00],
        [1.0000e+00, 1.0000e-01, 1.0000e-02, 1.0000e-03],
        [2.0000e+00, 2.0000e-01, 2.0000e-02, 2.0000e-03],
        [3.0000e+00, 3.0000e-01, 3.0000e-02, 3.0000e-03]])
torch.Size([4, 4])


In [48]:
def apply_rope(q, angles):
    B, T, d = q.shape

    # 1. 将最后一维拆成二维坐标对
    q_pairs = q.view(B,T,d//2,2)               # (B,T,d//2,2)

    # 2. 取出每对坐标的 x 和 y
    x = q_pairs[..., 0]           # (B,T,d//2)
    y = q_pairs[..., 1]           # (B,T,d//2)

    # 3. 计算旋转角度的 sin、cos
    cos = torch.cos(angles).to(dtype=q.dtype)
    sin = torch.sin(angles).to(dtype=q.dtype)

    # 4. 应用二维旋转公式
    x_rot = x * cos - y * sin
    y_rot = x * sin + y * cos

    # 5. 将旋转后的坐标重新组合
    q_rot = torch.stack([x_rot, y_rot], dim=-1)

    # 6. 还原成 (B,T,d)
    return q_rot.flatten(2)

In [49]:
q = torch.randn(2, 4, 8)
angles = get_rope_angles(4, 8)

q_rot = apply_rope(q, angles)

print(q_rot.shape)
print(torch.allclose(
    q.norm(dim=-1),
    q_rot.norm(dim=-1),
    atol=1e-5
))

torch.Size([2, 4, 8])
True


In [50]:
torch.manual_seed(42)

q = torch.randn(1, 1, 8)
k = torch.randn(1, 1, 8)

angles = get_rope_angles(10, 8)

# 第一组：Q 位于 2，K 位于 5
q_2 = apply_rope(q, angles[2:3])
k_5 = apply_rope(k, angles[5:6])

score_1 = torch.sum(q_2 * k_5)

# 第二组：Q 位于 5，K 位于 8
# TODO：你自己完成

q_5 = apply_rope(q, angles[5:6])
k_8 = apply_rope(k, angles[8:9])

score_2 = torch.sum(q_5 * k_8)

print(score_1.item())
print(score_2.item())
print(torch.allclose(score_1, score_2, atol=1e-5))

-3.708029270172119
-3.708029270172119
True


In [51]:
import sys
import subprocess

test = r"""
import torch
import sympy

print("Torch:", torch.__version__)
print("SymPy:", sympy.__version__)
print("sympy.core available:", hasattr(sympy, "core"))

p = torch.nn.Parameter(torch.randn(2, 2))
optimizer = torch.optim.AdamW([p], lr=3e-4)

print("AdamW OK!")
"""

result = subprocess.run(
    [sys.executable, "-c", test],
    capture_output=True,
    text=True
)

print("Return code:", result.returncode)
print(result.stdout)
print(result.stderr[-4000:])

Return code: 0
Torch: 2.14.1+cu130
SymPy: 1.14.0
sympy.core available: True
AdamW OK!




# 以下是实验

In [59]:
import torch
import tiktoken
import torch.nn as nn
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
from torch.nn import functional as F

# hyperparameters
batch_size = 8      # 保持
block_size = 256      # 从 256 降到 128（计算量降 4 倍）
max_iters = 5000     # 先跑 1000 步
eval_interval = 200   # 每 100 步才评估一次（拉长间隔）
learning_rate = 3e-4  # 保持
eval_iters = 20       # 从 200 降到 50（评估快 4 倍）
n_embd = 384          # 保持
n_head = 6            # 保持
n_layer = 12           # 保持
dropout = 0.1         # 保持
device = 'cuda' if torch.cuda.is_available() else 'cpu'
# ------------

torch.manual_seed(1337) #for replication


with open('input.txt', 'r', encoding='utf-8') as f: #讀一讀
    text = f.read()
enc = tiktoken.get_encoding("cl100k_base")

# 原始 Token IDs
original_ids = torch.tensor(enc.encode(text), dtype=torch.long)

# 建立紧凑词表：旧 ID -> 新 ID
unique_ids, data = torch.unique(
    original_ids,
    sorted=True,
    return_inverse=True
)

vocab_size = len(unique_ids)

def encode(s):
    ids = torch.tensor(enc.encode(s), dtype=torch.long)
    positions = torch.searchsorted(unique_ids, ids)

    valid = positions < vocab_size
    valid[valid.clone()] &= (
        unique_ids[positions[valid.clone()]] == ids[valid.clone()]
    )

    if not valid.all():
        raise ValueError("输入包含训练语料词表以外的 Token")

    return positions.tolist()

def decode(ids):
    original = unique_ids[
        torch.tensor(ids, dtype=torch.long)
    ].tolist()
    return enc.decode(original)

print("New vocabulary size:", vocab_size)
print("New token IDs:", data[:20])
n = int(0.9*len(data)) # 得到長度
train_data = data[:n] #劃分訓練集
val_data = data[n:] #劃分驗證集
print("Number of tokens:", len(data))
print("Vocabulary size:", vocab_size)
print("Unique tokens in corpus:", len(torch.unique(data)))

#
def get_batch(split):
    # 根據split是啥進行操作
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    #torch.randint的語法,前者是最大值 至少抽出來之後能往後走block_size個；
    #(batch_size，)就是輸出的數據長什麽樣子啦 抽出來batch_size組扔進去
    x = torch.stack([data[i:i+block_size] for i in ix])
    #去data這個list[int]裏面把i到i+blocksize-1這些都拿出來

    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    #去data這個list[int]裏面把i+1到i+blocksize這些都拿出來

    #stack就是曡羅漢 把抓出來的batch_size個長度爲block_size的tensor堆叠
    #變成[B,T]
    x, y = x.to(device), y.to(device)
    return x, y #兩個[B,T]

@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()

    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)

        for k in range(eval_iters):
            X, Y = get_batch(split)

            with torch.autocast(
                device_type='cuda',
                dtype=torch.bfloat16,
                enabled=(device == 'cuda')
            ):
                logits, loss = model(X, Y)

            losses[k] = loss.item()

        out[split] = losses.mean()

    model.train()
    return out

class Head(nn.Module):
    """ one head of self-attention """

    def __init__(self, head_size):
        super().__init__() #繼承nnModule
        self.key = nn.Linear(n_embd, head_size, bias=False)
        #[n_embd, head_size]的矩陣綫性層
#我們投喂的x 其實這個時候channel的C已經是表徵空間了
#但我們不打算直接用它 而是每個head都學習一套這個表徵的某一個側面的qkv
#然後把channel原來的n_embd長度改成head_size
        self.query = nn.Linear(n_embd, head_size, bias=False)
        #同上
        self.value = nn.Linear(n_embd, head_size, bias=False)
        #同上
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))
        #製造一個[T,T]大小的下三矩陣
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B,T,C = x.shape
        k = self.key(x) #[B,T,H]
        q = self.query(x) #[B,T,H]

        angles = get_rope_angles(T, q.shape[-1]).to(x.device)
        q = apply_rope(q, angles)
        k = apply_rope(k, angles)
        wei = q @ k.transpose(-2, -1) * q.shape[-1]**-0.5
        #[B,T,H]@[B,H,T] = [B,T,T]
        #這樣就真的是每個token對每個token的打分了
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        #masked_fill做掩碼 把滿足要求的都改成負無窮
        wei = F.softmax(wei, dim=-1) #做softmax
        wei = self.dropout(wei) ## 注意力 dropout
        v = self.value(x) #把[B,T,H]提取
        out = wei @ v #[B,T,T] @ [B,T,H] = [B,T,H]
        return out

class MultiHeadAttention(nn.Module):
    """ multiple heads of self-attention in parallel """

    def __init__(self, num_heads, head_size):#自定義需要head數量+head長度
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        #全部放在一個list裏面一步一步來 注冊子模塊 追蹤參數
        self.proj = nn.Linear(n_embd, n_embd) # 1！！！
        #反正再映射一層
        self.dropout = nn.Dropout(dropout)
        #dropout一手

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        #每次從list裏面取出一個函數算一遍 輸出一個[B,T,H]*N就是[B,T,H*N]
        #不過在這份代碼裏面H*N=C是hardcode在裏面了 1！！！那個地方
        out = self.dropout(self.proj(out))
        return out #反正吐出來的是[B,T,C]

class FeedFoward(nn.Module):
    """ a simple linear layer followed by a non-linearity """

    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),#放大四倍
            nn.ReLU(),#非綫性層
            nn.Linear(4 * n_embd, n_embd),#恢復原先參數量級
            nn.Dropout(dropout),#drop一手
        ) #反正sequential就是組裝一些基本block

    def forward(self, x):
        return self.net(x) #again Module的定義裏面直接調用feedforward

class Block(nn.Module):
    """ Transformer block: communication followed by computation """

    def __init__(self, n_embd, n_head):#需要C和N
        super().__init__()
        head_size = n_embd // n_head #這裏是這樣定義的 但也不一定
        self.sa = MultiHeadAttention(n_head, head_size) #寫過了
        self.ffwd = FeedFoward(n_embd) #寫過了
        self.ln1 = nn.LayerNorm(n_embd) #寫過了
        self.ln2 = nn.LayerNorm(n_embd) #寫過了

    def forward(self, x):
        x = x + self.sa(self.ln1(x))  #layernorm+resnet處理梯度
        x = x + self.ffwd(self.ln2(x)) #layernorm+resnet處理梯度
        return x             #ln + sa +ln + ffwd

class BigramLanguageModel(nn.Module): #好的我們重寫這個大類

    def __init__(self):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd) #每個詞的表徵空間
        self.blocks = nn.Sequential(*[Block(n_embd, n_head=n_head) for _ in range(n_layer)])
        #跑n_layer輪的Block 然後用*解開list放到sequential裏
        self.ln_f = nn.LayerNorm(n_embd) #做一層ln
        self.lm_head = nn.Linear(n_embd, vocab_size) #把每個token的維度表示映射成vocab size上的打分

    def forward(self, idx, targets=None):
        B, T = idx.shape
        tok_emb = self.token_embedding_table(idx) #查表 把每個字符的對應的表徵空間
        x = tok_emb  #RoPE修正：不用直接加position embedding了
        x = self.blocks(x) #跑一遍blocks
        x = self.ln_f(x) #跑一邊ln
        logits = self.lm_head(x) #把每個token的維度表示映射成vocab size上的打分

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C) #展開 攤平batch 每個位置的logits
            targets = targets.view(B*T) #打扁平了
            loss = F.cross_entropy(logits, targets) #經典交叉熵搭配softmax計算loss

        return logits, loss #把logits吐出來 loss放出來

    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            # 1. 裁剪上下文
            idx_cond = idx[:, -block_size:]

            # 2. 前向传播
            logits, loss = self(idx_cond)

            # 3. 只取最后一个时间步的 logits (形状变成 [B, vocab_size])
            logits = logits[:, -1, :]

            # ==================== 采样策略 ====================
            # 4. 温度缩放 (除以 1.0，如果想要更具创造力，可以改成 1.1 或 1.2)
            logits = logits / 1.0

            # 5. Top-K 截断（只保留前 50 个最高分的词）
            top_k = 50
            v, _ = torch.topk(logits, top_k)
            min_val = v[:, -1].unsqueeze(-1)
            logits = torch.where(logits < min_val, torch.tensor(float('-inf')).to(device), logits)

            # 6. 再算 softmax
            probs = F.softmax(logits, dim=-1)
            # =================================================

            # 7. 按照概率抽一个
            idx_next = torch.multinomial(probs, num_samples=1)

            # 8. 拼接进去新的内容
            idx = torch.cat((idx, idx_next), dim=1)

        return idx

model = BigramLanguageModel() #instantiation
m = model.to(device) #把模型搬運到指定的設備
#
print(sum(p.numel() for p in m.parameters())/1e6, 'M parameters') #告訴我們有多少個參數

#
# 優化器
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)
for step in range(max_iters):

    if step == 1500:
        for param_group in optimizer.param_groups:
            param_group['lr'] = 1e-4

    if step % eval_interval == 0 or step == max_iters - 1:
        losses = estimate_loss()
        print(
            f"step {step}: "
            f"train loss {losses['train']:.4f}, "
            f"val loss {losses['val']:.4f}"
        )

    xb, yb = get_batch('train')

    # BF16 Forward
    with torch.autocast(
        device_type='cuda',
        dtype=torch.bfloat16,
        enabled=(device == 'cuda')
    ):
        logits, loss = model(xb, yb)

    # Backward
    optimizer.zero_grad(set_to_none=True)
    loss.backward()

    # Update
    optimizer.step()

# 生成测试
prompt = "两仪式看着黑桐干也，说道："
context = torch.tensor([enc.encode(prompt)], dtype=torch.long, device=device)
print(decode(m.generate(context, max_new_tokens=500)[0].tolist()))

New vocabulary size: 1304
New token IDs: tensor([ 580,  480,  112,  659,  875,  740,   81, 1280,  299,  399,  550,  659,
         875,  740,   81, 1280,  299,  376,  550,  659])
Number of tokens: 546885
Vocabulary size: 1304
Unique tokens in corpus: 1304
22.283288 M parameters
step 0: train loss 7.3099, val loss 7.3035
step 200: train loss 4.0071, val loss 4.3364
step 400: train loss 3.6474, val loss 4.1469
step 600: train loss 3.3795, val loss 3.9575
step 800: train loss 3.1766, val loss 3.8893
step 1000: train loss 2.9257, val loss 3.8336
step 1200: train loss 2.7943, val loss 3.7957
step 1400: train loss 2.6035, val loss 3.7637


KeyboardInterrupt: 

In [61]:
prompt = "两仪式看着黑桐干也，说道："
context = torch.tensor([enc.encode(prompt)], dtype=torch.long, device=device)
print(decode(m.generate(context, max_new_tokens=500)[0].tolist()))

AcceleratorError: CUDA error: device-side assert triggered
Search for `cudaErrorAssert' in https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__TYPES.html for more information.
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
The CUDA driver logged these messages, which may provide useful details:
Sticky error detected
Returning 710 (CUDA_ERROR_ASSERT) from cuStreamIsCapturing



In [62]:
print("vocab_size:", vocab_size)
print("embedding size:", model.token_embedding_table.num_embeddings)

prompt = "两仪式看着黑桐干也，说道："

ids = enc.encode(prompt)
print("Original Token IDs:", ids)
print("Max ID:", max(ids))
print("Valid for model:",
      max(ids) < model.token_embedding_table.num_embeddings)

vocab_size: 1304
embedding size: 1304
Original Token IDs: [78640, 6271, 103, 29430, 52030, 84949, 222, 57752, 44559, 238, 7190, 110, 75863, 3922, 37687, 45893, 5232]
Max ID: 84949
Valid for model: False
